<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la ruta de rescate docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: Un placebo o control negativo y una frase que diga qué no demuestra sobre el supuesto no observable.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la ruta de rescate docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: Qini, varios placebos, E-value y R² parcial.
<!-- OPS:MAT-02:fin -->

# L11 · Validación: Qini, placebo y sensibilidad

**S11 · Lectura: Cap. 14 · 40–55 min**

## La pregunta

¿Qué prueba podría hacer que dejemos de confiar en el estimador?

## Qué harás

Validar un efecto heterogéneo sin confundir un placebo con una prueba total.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Qini, placebo, control negativo y sensibilidad del estimador A2.

> **Idea clave:** Validar no es buscar una cifra bonita; es buscar fallos plausibles.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
from lab_assumptions import rr_from_d
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
rng = setup()
N_PERM = R(30, 200)
print("permutaciones del placebo:", N_PERM)

## 1. Evaluar un CATE: el problema (§14.1–14.2)
Con datos de un **experimento** podemos evaluar un ranking de CATE sin conocer $\tau_i$: el *resultado transformado*
$$Y^*_i=Y_i\,\frac{T_i-e}{e(1-e)}\quad\Rightarrow\quad E[Y^*\mid X]=\tau(X)$$
(con $e$ conocido). Ordenamos las unidades de mayor a menor $\hat\tau$, acumulamos $Y^*$ y comparamos con la línea del **azar**: la curva de **Qini**.

Mundo simulado (el del capítulo): $\tau(X)=2+3X_0-X_1^2$, experimento con $e=0{,}5$.

In [ ]:
def make_dgp(n=3000, p=10, rng=None):
    rng = rng or np.random.default_rng(0)
    X = rng.normal(size=(n, p)); tau = 2 + 3 * X[:, 0] - X[:, 1] ** 2
    t = rng.binomial(1, .5, n); mu0 = 1.0 * X[:, 0] + X[:, 2] + .5 * X[:, 3] ** 2
    y = mu0 + t * tau + rng.normal(size=n)
    return X, y, t, tau, mu0
X, Y, T, tau, mu0 = make_dgp(3000, 10, np.random.default_rng(1))
tr, te = np.arange(2000), np.arange(2000, 3000)
mk = lambda: ExtraTreesRegressor(n_estimators=60, min_samples_leaf=10, max_features=0.8, random_state=0)
# T-learner entrenado SOLO en train; se evalúa en test
tau_hat = mk().fit(X[tr][T[tr] == 1], Y[tr][T[tr] == 1]).predict(X[te]) - mk().fit(X[tr][T[tr] == 0], Y[tr][T[tr] == 0]).predict(X[te])
print("τ̂ en test: media", tau_hat.mean().round(2), "· desv.", tau_hat.std().round(2))

### Completa tú: el área de Qini
`qini_area(tau_hat, y, t, e)` debe: (1) calcular $Y^*$; (2) ordenar por $\hat\tau$ **descendente**; (3) acumular $Y^*$ y dividir por $n$ (uplift acumulado); (4) restar la **línea del azar** (de 0 al total, proporcional a la fracción tratada); (5) devolver el promedio de esa diferencia (área entre la curva y el azar).

In [ ]:
# ✏️ Completa tú — qini_area
# Pista: Un buen ranking pone arriba a quienes más se benefician: la curva queda POR ENCIMA del azar.
def qini_area(tau_hat, y, t, e=0.5):
    """Área (promedio) entre la curva de Qini y la línea del azar."""
    y, t = np.asarray(y, float), np.asarray(t, float)
    # TODO 1: resultado transformado y* = y (t-e) / (e (1-e))
    # TODO 2: ordena por tau_hat descendente y acumula y* / n
    # TODO 3: línea del azar = fracción acumulada * total
    raise NotImplementedError("completa qini_area")

In [ ]:
r_ = np.random.default_rng(0); t_s = r_.binomial(1, .5, 2000); y_s = r_.normal(size=2000) + t_s * 2 * (np.arange(2000) < 1000)
bueno = (np.arange(2000) < 1000).astype(float)            # ranking perfecto: los 1000 primeros tienen efecto 2
lab_check("qini_area (propiedades)", lambda: (qini_area(bueno, y_s, t_s) > 0.2, qini_area(-bueno, y_s, t_s) < -0.2,
          abs(qini_area(r_.normal(size=2000), y_s, t_s)) < 0.25), [((), (True, True, True), 0)])

In [ ]:
q_modelo = qini_area(tau_hat, Y[te], T[te])
q_azar = np.mean([qini_area(np.random.default_rng(s).normal(size=len(te)), Y[te], T[te]) for s in range(50)])
print(f"Qini del modelo: {q_modelo:.3f} · Qini de un ranking al azar (50 repeticiones): {q_azar:.3f}")
def curva(score):
    ys = Y[te] * (T[te] - .5) / .25; o = np.argsort(-score); return np.cumsum(ys[o]) / len(te)
f = np.arange(1, len(te) + 1) / len(te)
plt.plot(f, curva(tau_hat), label="modelo"); plt.plot(f, f * curva(tau_hat)[-1], "k--", label="azar")
plt.xlabel("fracción tratada (de mayor a menor τ̂)"); plt.ylabel("uplift acumulado"); plt.legend(); plt.show()

## 2. Baselines obligatorios (§14.3)
Un CATE solo "vale" si supera alternativas triviales:
1. **Efecto constante**: todos reciben el ATE ⇒ ranking sin información (área ≈ 0).
2. **Solo resultado**: ordenar por $\hat\mu_0(X)$ (quién tendría el mejor resultado sin tratamiento); es lo que hace un modelo predictivo sin causalidad.

In [ ]:
mu0_hat = mk().fit(X[tr][T[tr] == 0], Y[tr][T[tr] == 0]).predict(X[te])
filas = {"efecto constante (ATE)": qini_area(np.full(len(te), tau_hat.mean()) + 1e-9 * np.arange(len(te)), Y[te], T[te]),
         "solo resultado (ordena por μ̂₀)": qini_area(mu0_hat, Y[te], T[te]),
         "modelo (T-learner)": q_modelo}
# EVALUATION-CELL: techo con el τ verdadero (solo existe en simulación)
filas["oráculo (τ verdadero)"] = qini_area(tau[te], Y[te], T[te])
show(pd.Series(filas, name="área de Qini").round(3).to_frame())

**Mira el resultado.** El baseline «solo resultado» sale con un área positiva. ¿Significa que predecir el resultado sirve para decidir a quién tratar?

**Respuesta:**

Aquí sale positiva porque en este mundo $\mu_0$ y $\tau$ comparten la covariable $X_0$ (lo diseñamos así), y aun así queda muy por debajo del modelo causal. Con otro mundo —donde $\mu_0$ crezca justo donde $\tau$ baja— el área sería **negativa**. Ordenar por el resultado esperado sin tratamiento **no** es ordenar por beneficio: coincide con el CATE solo si ambos dependen de las mismas covariables y en el mismo sentido. Un modelo predictivo muy bueno puede ser inútil —o dañino— para decidir a quién tratar. Un CATE que no mejora a este baseline no justifica su complejidad.



## 3. Placebo por permutación con distribución nula (§14.4)
Idea: si **permutamos** el tratamiento entre las unidades, el verdadero efecto desaparece. Un pipeline sano debe devolver ≈ 0. Una sola permutación es una anécdota: repetimos `N_PERM` veces y obtenemos una **distribución nula**, refitando **todo** el pipeline (nuisance incluidos) en cada una.

Usamos un PLR con cross-fitting (K = 3, ExtraTrees pequeños) para que repetirlo sea rápido.

In [ ]:
fast = lambda: ExtraTreesRegressor(n_estimators=15, min_samples_leaf=10, max_features=0.8, random_state=0)
def plr_fast(X_, y_, t_, K=3, seed=SEED):
    n = len(y_); ry = np.zeros(n); rd = np.zeros(n)
    for te_ in kfold(n, K, seed):
        tr_ = np.setdiff1d(np.arange(n), te_)
        ry[te_] = y_[te_] - fast().fit(X_[tr_], y_[tr_]).predict(X_[te_])
        rd[te_] = t_[te_] - fast().fit(X_[tr_], t_[tr_]).predict(X_[te_])
    th = (rd * ry).sum() / (rd * rd).sum(); psi = (ry - th * rd) * rd / np.mean(rd ** 2)
    return th, np.sqrt(np.mean(psi ** 2) / n)
Xi, Yi, Ti = (a[:1500] for a in (X, Y, T))
th_obs, se_obs = plr_fast(Xi, Yi, Ti)
print("estimación real :", fmt_ci(th_obs, se_obs), "  (ATE verdadero =", round(tau.mean(), 2), ")")
r_ = np.random.default_rng(11)
nulo = np.array([plr_fast(Xi, Yi, r_.permutation(Ti))[0] for _ in range(N_PERM)])
p_emp = (1 + (np.abs(nulo) >= abs(th_obs)).sum()) / (1 + len(nulo))
print(f"placebo: media {nulo.mean():+.3f} · desv. {nulo.std():.3f} · rango [{nulo.min():+.2f}, {nulo.max():+.2f}]")
print(f"p empírico de la estimación real frente a la nula: {p_emp:.3f}")
plt.hist(nulo, bins=20); plt.axvline(th_obs, c="r"); plt.axvline(0, c="k", ls="--"); plt.xlabel("θ̂ con tratamiento permutado (rojo: estimación real)"); plt.show()

### Lo que el placebo **no** demuestra
El placebo comprueba que el pipeline no *inventa* efectos. **No** detecta confusión no medida: si el estimador está sesgado por un confusor, al permutar $T$ el sesgo desaparece igual y el placebo "pasa". Lo vemos con `latent_confounder` ($T=X_8$, $Y=X_9$, confusor latente $X_0$, efecto verdadero conocido).

In [ ]:
lc = load_scenario("synthetic/edge_cases/latent_confounder"); lc.card()
Xl, Yl, Tl = lc.xyt("train")
th_l, se_l = plr_fast(Xl, Yl, Tl)
r_ = np.random.default_rng(12)
nulo_l = np.array([plr_fast(Xl, Yl, r_.permutation(Tl))[0] for _ in range(max(15, N_PERM // 2))])
print("estimación sin el confusor (X0 no observado):", fmt_ci(th_l, se_l))
print(f"placebo: media {nulo_l.mean():+.3f} · desv. {nulo_l.std():.3f}  → el placebo pasa (≈ 0)")

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

In [ ]:
# EVALUATION-CELL
print("efecto verdadero:", round(lc.true_ate, 3), "· sesgo de la estimación:", round(th_l - lc.true_ate, 3))
if lc.full is not None:
    cols_or = [c for c in lc.full.columns if c not in ("row_id", lc.treatment, lc.outcome)]
    d = lc.full.iloc[:len(Yl)]
    th_or, se_or = plr_fast(d[cols_or].to_numpy(float), d[lc.outcome].to_numpy(float), d[lc.treatment].to_numpy(float))
    print("con el confusor X0 disponible (oráculo):", fmt_ci(th_or, se_or))

**Mira el resultado.** El placebo «pasó» y la estimación está sesgada. ¿Qué diferencia hay entre «no hay evidencia de problema» y «el estimador es válido»?

**Respuesta:**

El placebo solo falla si el pipeline produce efectos donde no los hay (fugas, errores de código, varianza mal estimada). La confusión no medida actúa **solo cuando el tratamiento real está en juego**, así que la permutación la borra. Una prueba que no rechaza es compatible con un estimador válido **y** con uno sesgado: para lo segundo hace falta sensibilidad (E-value) o conocimiento externo.



## 4. E-value: cuánta confusión haría falta (§14.5, Def. 14.3)
Para una razón de riesgos $RR\ge1$:
$$E=RR+\sqrt{RR\,(RR-1)}.$$
Es la fuerza mínima (en escala RR, con tratamiento **y** resultado) que un confusor no medido necesitaría para **explicar por completo** la asociación. Con resultado continuo se usa la aproximación $RR\approx\exp(0{,}91\,d)$ con $d=\theta\,/\,\sigma_Y$ (diferencia estandarizada).

> **Aviso de escala.** El E-value está definido para exposición binaria. Con tratamiento continuo hay que fijar un contraste (aquí **+1 desviación estándar**) y decirlo. La aproximación $\exp(0{,}91d)$ es de VanderWeele y Ding (2017); conviene verificarla en la fuente antes de citarla.

### Completa tú
`e_value(rr)`: tratar $RR<1$ con su inverso. Casos: $RR=1\to1$; $RR=2\to3{,}414$.

In [ ]:
# ✏️ Completa tú — e_value
# Pista: Si RR < 1, usa 1/RR primero.
def e_value(rr):
    """E-value de VanderWeele y Ding para una razón de riesgos rr."""
    raise NotImplementedError("completa e_value")

In [ ]:
casos_ev = [((1.0,), 1.0, 1e-9), ((2.0,), 2 + np.sqrt(2), 1e-9), ((0.5,), 2 + np.sqrt(2), 1e-9)]
lab_check("e_value", e_value, casos_ev)

# --- Prueba de mutación: ¿tu comprobación detectaría una fórmula equivocada?
def pasa(fn, casos): return all(np.allclose(fn(*a), esp, atol=tol) for a, esp, tol in casos)
mutante_sin_raiz = lambda rr: rr + rr * (rr - 1)          # olvida la raíz cuadrada
print("¿tus casos detectan el mutante sin raíz?", "sí ✔" if not pasa(mutante_sin_raiz, casos_ev) else "NO ✘ (añade un caso)")

In [ ]:
sdT, sdY = Tl.std(), Yl.std()
def e_de(theta, se):
    d = abs(theta) * sdT / sdY                          # contraste: +1 DE del tratamiento
    lim = min(abs(theta - 1.96 * se), abs(theta + 1.96 * se)) if np.sign(theta - 1.96 * se) == np.sign(theta + 1.96 * se) else 0
    d_lim = lim * sdT / sdY
    return d, rr_from_d(d), e_value(rr_from_d(d)), e_value(rr_from_d(d_lim)) if lim > 0 else 1.0
d, rr, ev, ev_ic = e_de(th_l, se_l)
print(f"efecto por +1 DE de X8: d = {d:.3f} · RR ≈ {rr:.2f} · E-value = {ev:.2f} · E-value del límite del IC más cercano a 0 = {ev_ic:.2f}")

In [ ]:
# EVALUATION-CELL: ¿qué parte de la estimación es sesgo por el confusor latente?
bias = th_l - lc.true_ate
d_bias = abs(bias) * sdT / sdY
print(f"sesgo = {bias:+.3f} · en las unidades del E-value: {d_bias:.3f} DE de Y por +1 DE de X8 (la estimación equivale a {d:.3f})")
print("→ el confusor latente real " + ("SÍ tiene fuerza suficiente para explicar por completo la estimación." if d_bias >= d else "explica solo una parte de la estimación."))

**Mira el resultado.** ¿Un E-value «alto» garantiza que no hay confusión? ¿Qué decisión informa?

**Respuesta:**

No: dice **cuánta** confusión haría falta, no si existe. Es una herramienta para razonar: se compara con las asociaciones que se conocen de confusores plausibles (en la escala RR). Si ningún confusor conocido alcanza ese valor, el resultado es más creíble; si uno basta, hay que desconfiar. Su valor es obligar a *declarar* el supuesto no observable y acotarlo.



## 5. Tabla de refutaciones (§14.6)
Reunimos varias pruebas sobre el mismo estimador (el PLR rápido del experimento simulado, ATE verdadero ≈ 1). Cada fila **puede refutar, ninguna confirma**.

In [ ]:
rr_ = np.random.default_rng(21)
def fila(nombre, th, se, esperado, criterio):
    ok = criterio(th, se); return {"prueba": nombre, "θ̂": round(th, 3), "IC95": f"[{th - 1.96 * se:.2f}; {th + 1.96 * se:.2f}]", "esperado": esperado, "resultado": "pasa" if ok else "FALLA"}
ref = []
ref.append(fila("estimación real", th_obs, se_obs, "≠ 0", lambda t, s: abs(t) > 1.96 * s))
tp, sp = plr_fast(Xi, Yi, rr_.permutation(Ti)); ref.append(fila("placebo: tratamiento permutado", tp, sp, "≈ 0", lambda t, s: abs(t) < 1.96 * s))
Xr = np.column_stack([Xi, rr_.normal(size=len(Xi))]); tr_, sr_ = plr_fast(Xr, Yi, Ti)
ref.append(fila("causa común aleatoria añadida", tr_, sr_, "≈ estimación real", lambda t, s: abs(t - th_obs) < 1.96 * se_obs))
idx = rr_.choice(len(Yi), int(.8 * len(Yi)), replace=False); ts_, ss_ = plr_fast(Xi[idx], Yi[idx], Ti[idx])
ref.append(fila("submuestra del 80 %", ts_, ss_, "≈ estimación real", lambda t, s: abs(t - th_obs) < 1.96 * se_obs))
to_, so_ = plr_fast(Xi, Xi[:, 5] + rr_.normal(size=len(Yi)), Ti)
ref.append(fila("resultado placebo (Y no afectado por T)", to_, so_, "≈ 0", lambda t, s: abs(t) < 1.96 * s))
show(pd.DataFrame(ref).set_index("prueba"))

## 6. Cerrar A2: guardar el artefacto con su hash (§14.7)
La evaluación debe ser **reproducible**: se guarda la estimación junto con el *hash* de los datos de entrenamiento, para que cualquier cambio posterior se detecte. Ejecuta A2 sobre IHDP-medium.

In [ ]:
art = a2_fit(save=True)
print("hash de los datos de entrenamiento:", art["hash_datos_entrenamiento"])
print("θ̂ =", fmt_ci(art["theta"], art["se"]), "· n =", art["n_train"], "· covariables:", len(art["covariables"]))

In [ ]:
# EVALUATION-CELL
sc = load_scenario("synthetic/inference/ihdp_medium"); print("ATE verdadero de IHDP-medium:", round(sc.true_ate, 3))

## 7. Supuesto y bitácora

In [ ]:
fila_b = bitacora(
    pregunta="¿Supera el CATE los baselines y resiste los placebos? ¿Qué no demuestra que el placebo no rechace?",
    metodo="Qini con resultado transformado; placebo por permutación (distribución nula); E-value del +1 DE de X8",
    estimando="Calidad del ranking de CATE (área de Qini) y robustez del ATE",
    numero_ic=f"Qini {q_modelo:.3f} frente a azar {q_azar:.3f}; placebo media {nulo.mean():+.3f} (±{nulo.std():.3f}); E-value {ev:.2f}",
    supuesto=supuesto("falsificable", "contrastado", "el placebo no rechaza, pero eso NO demuestra ausencia de confusión no medida (el E-value acota cuánta haría falta)"),
    decision="continuar", lab="L11")

---
## Qué queda para el libro
Quedan en el libro: RATE, la evaluación versionada con ledger y el protocolo completo de sensibilidad de tres niveles (aquí solo el E-value). La clave exacta de `placebo_permute_treatment` del repositorio no se usa: el lab hace su propia permutación.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*